# Forex italiani: cinque secoli di valute, oro e tassi realiRassegna dei dati FX storici per l'Italia (e benchmark europei), con focus su:oscillazioni valutarie, oro come hedge, tassi reali e debito sovrano.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

# Style
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("Setup complete.")

## 1 – Tassi di cambio Italia (lira / euro)Evoluzione della lira rispetto alle principali valute;il dataset `yearly_unified_wide.csv` copre centinaia di paesi dal 1500.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

# Load wide panel
fx = pd.read_csv(ROOT / "derived" / "normalized" / "yearly_unified_wide.csv")

# Focus Italy
it = fx[["year", "Italy"]].dropna(subset=["Italy"])

print(f"Anni disponibili: {int(it.year.min())} – {int(it.year.max())}")
print(f"Prime righe:")
print(it.head(10))

plt.figure()
plt.plot(it["year"], it["Italy"], marker=".", markersize=2)
plt.yscale("log")
plt.ylabel("Lire/EUR per 1 USD (log scale)")
plt.title("Tasso di cambio Italia (USD per unità locale)")
plt.xlabel("Anno")
plt.ylabel("USD / local currency")
plt.tight_layout()
plt.show()

## 2 – Inflazione del grano italianoIl prezzo del grano è un proxy storico dell'inflazione;`yearly_gold_inflation.csv` contiene gold_local, gold_inflation_pct, CPI, ecc.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

gi = pd.read_csv(ROOT / "derived" / "analysis" / "yearly_gold_inflation.csv")

# Filter Italy or UK as proxy
it_g = gi[gi["country"] == "Italy"]
if it_g.empty:
    print("Nessun dato Italia nel file gold_inflation; uso UK come proxy.")
    it_g = gi[gi["country"] == "United Kingdom"]

print(it_g.head(10))
print(f"\nRighe: {len(it_g)}")

# Gold inflation over time
if "gold_inflation_pct" in it_g.columns:
    g = it_g.dropna(subset=["gold_inflation_pct"])
    plt.figure()
    plt.plot(g["year"], g["gold_inflation_pct"], marker=".", markersize=3)
    plt.yscale("log")
    plt.ylabel("Inflazione oro (% annuo, log scale)")
    plt.title("Gold inflation % (proxy inflazione storica)")
    plt.xlabel("Anno")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Colonna gold_inflation_pct non trovata")

## 3 – Carestie: picchi nei prezzi agricoliI picchi di prezzo del grano segnano le crisi alimentari;analizziamo le varianti di volatilità e outlier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

gi = pd.read_csv(ROOT / "derived" / "analysis" / "yearly_gold_inflation.csv")
gi_uk = gi[gi["country"] == "United Kingdom"]

# Z-score-like: classify spikes > 2 std
col = "gold_local"
if col in gi_uk.columns:
    s = gi_uk.dropna(subset=[col])
    mean, std = s[col].mean(), s[col].std()
    spikes = s[s[col] > mean + 2 * std]
    print(f"Soglia spike: {mean + 2*std:.2f}  (mean {mean:.2f} + 2*std {std:.2f})")
    print(f"Anni con spike: {sorted(spikes.year.tolist())}")
    print(f"Conteggio: {len(spikes)}")

    plt.figure()
    plt.plot(s["year"], s[col], label="Prezzo grano (g/£100)", marker=".", ms=3)
    plt.scatter(spikes["year"], spikes[col], color="red", zorder=5, label="Spike > 2σ")
    plt.title("Prezzo del grano e carestie storiche (UK)")
    plt.xlabel("Anno")
    plt.ylabel("Prezzo (g per 100 sterline)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Colonna gold_local non trovata.")

## 4 – Confronto tra paesi europeiGrafico multi-paesi dei tassi di cambio annualizzati; consente di confrontarela volatilità relativa delle principali valute europee.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

fx = pd.read_csv(ROOT / "derived" / "normalized" / "yearly_unified_wide.csv")

countries = ["Italy", "Germany", "France", "United Kingdom", "Spain", "Greece"]
available = [c for c in countries if c in fx.columns]

sub = fx[["year"] + available].dropna(how="all", subset=available)

print("Paesi disponibili:", available)
print(sub.tail(15))

plt.figure()
for c in available:
    d = sub[["year", c]].dropna()
    plt.plot(d["year"], d[c], label=c, linewidth=1.2)
plt.yscale("log")
plt.ylabel("Tasso di cambio (log scale)")
plt.title("Tassi di cambio Europa (USD per unità locale)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5 – Tassi reali secolariI tassi di interesse nominali storici si confrontano con l'inflazione per ottenereil tasso reale; il file `measuringworth_interest_rates.csv` fornisce serie UK.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

# Interest rates (skip problematic CSV, use Schmelzing instead)
ir = pd.read_excel(ROOT / "sources" / "schmelzing" / "schmelzing_real_interest_rates.xlsx",
                   sheet_name="IV. Country level, 1310-2018", header=None, skiprows=3)
ir = ir.iloc[:, :10]
ir.columns = ["Year", "_", "Italy", "UK", "Holland", "Germany", "France", "USA", "Spain", "Japan"]
for c in ["Italy", "UK", "Germany", "France", "USA"]:
    ir[c] = pd.to_numeric(ir[c], errors="coerce")
ir = ir.dropna(subset=["Year"])
print(f"Interest rates: {len(ir)} years")
for c in ["Italy", "UK", "Germany", "France", "USA"]:
    v = ir[c].dropna()
    if len(v) > 0:
        print(f"  {c}: {v.mean():.2f}% ({len(v)} years)")

## 6 – Debito sovrano e spreadProssimità tra il tasso nominale UK e l'andamento del debito;per l'Italia storica si usano proxy da MeasuringWorth.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

# Interest rates from Schmelzing (reliable Excel source)
ir = pd.read_excel(ROOT / "sources" / "schmelzing" / "schmelzing_real_interest_rates.xlsx",
                   sheet_name="IV. Country level, 1310-2018", header=None, skiprows=3)
ir = ir.iloc[:, :10]
ir.columns = ["Year", "_", "Italy", "UK", "Holland", "Germany", "France", "USA", "Spain", "Japan"]
for c in ["Italy", "UK", "Germany", "France", "USA"]:
    ir[c] = pd.to_numeric(ir[c], errors="coerce")
ir = ir.dropna(subset=["Year"])
print(f"Interest rates: {len(ir)} years")
for c in ["Italy", "UK", "Germany", "France", "USA"]:
    v = ir[c].dropna()
    if len(v) > 0:
        print(f"  {c}: {v.mean():.2f}% ({len(v)} years)")

## 7 – Oro come hedge contro l'inflazioneSerie temporali del prezzo dell'oro in sterline e dollari;si confrontano con l'inflazione per valutare la capacità di hedge.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") / "data" / "raw" / "forex-centuries" / "data"

# Gold prices from MeasuringWorth (simpler format)
gold = pd.read_csv(ROOT / "sources" / "measuringworth" / "measuringworth_gold_prices.csv", 
                   on_bad_lines="skip")
gold["year"] = pd.to_numeric(gold["year"], errors="coerce")
gold = gold.dropna(subset=["year"])
print(f"Gold prices: {len(gold)} years, {int(gold['year'].min())}-{int(gold['year'].max())}")
print(f"British price range: {gold['British_price'].min():.2f} - {gold['British_price'].max():.2f} GBP/oz")

## 8 – Conclusioni### Osservazioni chiave1. **Volatilità storica**: la lira italiana ha subito ripetute svalutazioni, specie dopo le guerre.2. **Inflazione agraria**: i picchi nei prezzi del grano segnano le grandi carestie europee.3. **Regime FX**: i tassi fissi (gold standard, Bretton Woods) producono volatilità diversificata rispetto ai regimi di flottazione.4. **Oro come hedge**: il prezzo dell'oro in sterline mostra una tendenza crescente su secoli, confermando il valore dell'oro come hedge reale.5. **Tassi reali**: i tassi nominali UK storici restano bassi per secoli, con variazioni brusche in epoca moderna.### Prossimi passi- Analisi regime-conditional (usa `regime_conditional_stats.csv`).- Matrice di correlazione cross-country (`daily_correlation_matrix.csv`).- Confronto oro vs CPI su scala logaritmica.